## 1. Setup and Installation

First, let's ensure all dependencies are installed and import the required modules.

In [1]:
# If running for the first time, uncomment and run:
# !pip install -q yfinance pandas jupyter

# Add the parent directory to Python path to import our module
import sys
sys.path.insert(0, '..')

from src import (
    StockFetcher,
    Stock,
    print_quote_summary,
    print_historical_summary,
    export_quote_to_csv,
    export_historical_to_csv,
    format_price,
    format_volume,
    format_percent
)

import pandas as pd
from datetime import datetime, timedelta

print("✓ All imports successful!")

✓ All imports successful!


## 2. Fetching Stock Quotes

Let's start by fetching current stock quotes for popular stocks.

In [2]:
# Initialize the fetcher
fetcher = StockFetcher()

# Fetch a single stock quote
print("Fetching AAPL stock quote...")
aapl = fetcher.fetch_quote("AAPL")

# Display the quote
print_quote_summary(aapl)

Fetching AAPL stock quote...

Stock Quote Summary - AAPL
Current Price:    $300.23
Open Price:       $297.75
High (Day):       $303.20
Low (Day):        $296.53
Previous Close:   $298.21
Price Change:     +2.02 (+0.68%)
Volume:           54,675,423
Currency:         USD
Timestamp:        2026-05-15 22:57:29.178124



In [3]:
# Access specific quote information
if aapl.quote:
    print(f"Stock: {aapl.symbol}")
    print(f"Current Price: {format_price(aapl.quote.price)}")
    print(f"Price Change: {format_price(aapl.get_price_change())} ({format_percent(aapl.get_price_change_percent())})")
    print(f"Today's High: {format_price(aapl.quote.high_price)}")
    print(f"Today's Low: {format_price(aapl.quote.low_price)}")
    print(f"Volume: {format_volume(aapl.quote.volume)}")

Stock: AAPL
Current Price: $300.23
Price Change: $2.02 (+0.68%)
Today's High: $303.20
Today's Low: $296.53
Volume: 54,675,423


## 3. Analyzing Historical Data

Now let's fetch and analyze historical price data.

In [4]:
# Fetch historical data for the last 3 months
print("Fetching 3 months of historical data for AAPL...")
aapl_history = fetcher.fetch_historical_data("AAPL", period="3mo", interval="1d")

# Display summary
print_historical_summary(aapl_history)

Fetching 3 months of historical data for AAPL...
Error fetching historical data for AAPL: 'Adj Close'
AAPL: No historical data available


In [5]:
# Convert historical data to pandas DataFrame for analysis
if aapl_history.historical_data:
    data = []
    for hd in aapl_history.historical_data:
        data.append({
            'Date': hd.date.date(),
            'Open': hd.open_price,
            'High': hd.high_price,
            'Low': hd.low_price,
            'Close': hd.close_price,
            'Volume': hd.volume,
            'Adj Close': hd.adjusted_close
        })
    
    df = pd.DataFrame(data)
    
    # Display first few rows
    print("\nFirst 5 rows of historical data:")
    print(df.head())
    print(f"\nTotal rows: {len(df)}")

In [6]:
# Calculate some basic statistics
if aapl_history.historical_data:
    df_copy = df.copy()
    df_copy['Daily Change'] = df_copy['Close'].diff()
    df_copy['Daily Change %'] = df_copy['Close'].pct_change() * 100
    
    print("Price Statistics:")
    print(f"Min Close: {format_price(df['Close'].min())}")
    print(f"Max Close: {format_price(df['Close'].max())}")
    print(f"Mean Close: {format_price(df['Close'].mean())}")
    print(f"Std Dev: {format_price(df['Close'].std())}")
    print(f"\nAverage Daily Volume: {format_volume(int(df['Volume'].mean()))}")
    print(f"Max Volume: {format_volume(int(df['Volume'].max()))}")

## 4. Tracking Multiple Stocks

Compare multiple stocks at once.

In [7]:
# Fetch quotes for multiple stocks
symbols = ["AAPL", "GOOGL", "MSFT", "AMZN", "TSLA"]
print(f"Fetching quotes for {symbols}...\n")

stocks = fetcher.fetch_multiple_quotes(symbols)

# Create a comparison table
stock_data = []
for symbol, stock in stocks.items():
    if stock.quote:
        stock_data.append({
            'Symbol': symbol,
            'Price': f"{stock.quote.price:.2f}",
            'Change': f"{stock.get_price_change():.2f}",
            'Change %': f"{stock.get_price_change_percent():.2f}%",
            'Open': f"{stock.quote.open_price:.2f}",
            'High': f"{stock.quote.high_price:.2f}",
            'Low': f"{stock.quote.low_price:.2f}",
            'Volume': f"{stock.quote.volume:,}"
        })

comparison_df = pd.DataFrame(stock_data)
print(comparison_df.to_string(index=False))

Fetching quotes for ['AAPL', 'GOOGL', 'MSFT', 'AMZN', 'TSLA']...

Symbol  Price Change Change %   Open   High    Low     Volume
  AAPL 300.23   2.02    0.68% 297.75 303.20 296.53 54,675,423
 GOOGL 396.78  -4.29   -1.07% 396.28 399.54 393.18 20,175,668
  MSFT 421.92  12.49    3.05% 414.41 428.16 412.92 50,237,244
  AMZN 264.14  -3.08   -1.15% 262.59 264.35 260.89 40,582,258
  TSLA 422.24 -21.06   -4.75% 434.10 434.66 422.00 52,085,091


## 5. Exporting Data to CSV

Save your stock data to CSV files for further analysis or record-keeping.

In [ ]:
# Fetch both quote and historical data
print("Fetching GOOGL quote and 6 months of historical data...")
googl = fetcher.fetch_quote_and_history("GOOGL", period="6mo", interval="1d")

# Export quote to CSV
quote_file = export_quote_to_csv(googl)
print(f"✓ Quote exported to: {quote_file}")

# Export historical data to CSV
history_file = export_historical_to_csv(googl)
print(f"✓ Historical data exported to: {history_file}")

In [ ]:
# Read back the exported CSV file
import os

if os.path.exists(history_file):
    df_exported = pd.read_csv(history_file)
    print(f"\nExported data (first 5 rows):")
    print(df_exported.head())
    print(f"\nTotal rows exported: {len(df_exported)}")

## 6. Advanced Analytics

Perform more advanced analysis on stock data.

In [ ]:
# Calculate moving average
if aapl_history.historical_data:
    df_analysis = df.copy()
    
    # 5-day and 20-day moving averages
    df_analysis['MA_5'] = df_analysis['Close'].rolling(window=5).mean()
    df_analysis['MA_20'] = df_analysis['Close'].rolling(window=20).mean()
    
    print("Moving Averages (Last 5 rows):")
    print(df_analysis[['Date', 'Close', 'MA_5', 'MA_20']].tail())

In [ ]:
# Calculate volatility
if aapl_history.historical_data:
    returns = df['Close'].pct_change()
    volatility = returns.std() * (252 ** 0.5)  # Annualized volatility
    
    print(f"\nAPPL Volatility Analysis:")
    print(f"Daily Volatility: {returns.std():.4f}")
    print(f"Annualized Volatility: {volatility:.4f} or {volatility*100:.2f}%")
    print(f"\nPrice Range (3 months):")
    print(f"Lowest: {format_price(df['Close'].min())}")
    print(f"Highest: {format_price(df['Close'].max())}")
    print(f"Current (from quote): {format_price(aapl.quote.price if aapl.quote else 0)}")

In [ ]:
# Year-to-date performance comparison
ytd_symbols = ["SPY", "QQQ", "DIA"]  # Market ETFs
print("Fetching YTD data for major indices...\n")

ytd_stocks = fetcher.fetch_multiple_historical(ytd_symbols, period="1y", interval="1d")

for symbol, stock in ytd_stocks.items():
    if stock.historical_data:
        hist_list = stock.historical_data
        start_price = hist_list[0].open_price
        end_price = hist_list[-1].close_price
        ytd_change = ((end_price - start_price) / start_price) * 100
        
        print(f"{symbol}:")
        print(f"  Start (1y ago): {format_price(start_price)}")
        print(f"  Current: {format_price(end_price)}")
        print(f"  1-Year Change: {format_percent(ytd_change)}\n")

## Summary

You've learned how to:
- ✓ Fetch current stock quotes
- ✓ Retrieve and analyze historical data
- ✓ Compare multiple stocks
- ✓ Export data to CSV
- ✓ Perform basic technical analysis
- ✓ Calculate volatility and moving averages

### Next Steps:
1. Explore different time periods and intervals
2. Analyze more stocks and sectors
3. Build your own analysis workflows
4. Create visualization dashboards

For more information, check the [README.md](../README.md) documentation.